# Multi-modal Fine-Tuning for Question Paper Generation

This notebook uses the `unsloth` library to fine-tune a Vision-Language Model on your specific historical question papers dataset, including embedded images and diagrams.

In [ ]:
!git clone https://github.com/Prince649294u83/QP.git
%cd QP/dsatm-qpgen-backend

In [ ]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes

In [ ]:
# Clear any leftover GPU memory from previous runs
import torch, gc
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()

from unsloth import FastVisionModel
max_seq_length = 1024 # Reduced for T4 15GB VRAM
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

model, tokenizer = FastVisionModel.from_pretrained(
    model_name = "unsloth/Llama-3.2-11B-Vision-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    load_in_4bit = load_in_4bit,
    use_gradient_checkpointing = "unsloth",
)

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    finetune_vision = True, # Finetune the vision encoder
    finetune_language = True, # Finetune the language model
    finetune_attention_modules = True, # Finetune attention
    finetune_mlp_modules = True, # Finetune MLP
    r = 16,
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

In [ ]:
import torch
from PIL import Image
import os
import json

# We use a custom PyTorch dataset to completely bypass the Hugging Face `datasets` library.
# Hugging Face `datasets` has a known bug where it flattens complex nested dictionaries (like vision messages)
# into dict-of-lists, which causes `KeyError: 0` in Unsloth's data collator.
class QPVisionDataset(torch.utils.data.Dataset):
    def __init__(self, jsonl_path):
        self.data = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                self.data.append(json.loads(line))
                
    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Load image if it exists
        images_paths = item.get("images", [])
        has_image = False
        loaded_images = []
        
        if images_paths and len(images_paths) > 0 and os.path.exists(images_paths[0]):
            has_image = True
            try:
                loaded_images = [Image.open(images_paths[0]).convert("RGB")]
            except:
                has_image = False
                loaded_images = []
                
        # Build the exact Llama 3.2 Vision format directly
        content = []
        if has_image:
            content.append({"type": "image"})
        content.append({"type": "text", "text": item["instruction"]})
        
        messages = [
            {"role": "user", "content": content},
            {"role": "assistant", "content": [{"type": "text", "text": item["output"]}]}
        ]
        
        return {
            "messages": messages,
            "images": loaded_images
        }

# Initialize the custom dataset
dataset = QPVisionDataset("dataset.jsonl")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth import is_bfloat16_supported
import unsloth

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Packing is not supported for vision models yet
    args = SFTConfig(
        per_device_train_batch_size = 1, # Set to 1 to avoid mixed modality batching errors in Llama 3.2 Vision
        gradient_accumulation_steps = 8, # Doubled to keep effective batch size = 8
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        remove_unused_columns=False,
        dataset_kwargs = {"skip_prepare_dataset": True},
    ),
    data_collator = unsloth.UnslothVisionDataCollator(model, tokenizer),
)

In [ ]:
trainer_stats = trainer.train()

In [ ]:
# 1. Save the LoRA adapters (lightweight, easy to load back into Unsloth)
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")
print("LoRA model saved to lora_model/")

In [ ]:
# 2. Export to GGUF (for Ollama / llama.cpp)
try:
    print("Exporting to GGUF format... This may take a while.")
    model.save_pretrained_gguf("gguf_model", tokenizer, quantization_method="q4_k_m")
    print("GGUF export complete!")
except Exception as e:
    print(f"GGUF export encountered an error: {e}")
    print("If GGUF fails, you can still use the lora_model saved above.")

In [ ]:
# 3. Zip and download the models to your local machine
import os
from google.colab import files

if os.path.exists("lora_model"):
    !zip -r lora_model.zip lora_model/
    files.download("lora_model.zip")

if os.path.exists("gguf_model"):
    !zip -r gguf_model.zip gguf_model/
    files.download("gguf_model.zip")
else:
    # Sometimes Unsloth saves GGUF files directly in the root directory instead of a folder
    # Let's search for .gguf files and download them
    import glob
    gguf_files = glob.glob("*.gguf")
    for file in gguf_files:
        print(f"Found GGUF file: {file}")
        files.download(file)
